# 2 · Heterogeneity  `[EVAL]`

Does the gain depend on **who the patient is**? Every metric split by true persona trait (recovered across the per-iteration shuffle). Two views per trait: a **combined all-metrics overview** (`2_heterogeneity/<trait>_all_metrics.png`) and the **per-metric subfolder** (`2_heterogeneity/<trait>/<metric>.png`, one panel per arm), plus the final-iteration endpoint bars. Traits: `cooperation_level` (Resistant / Cooperative / Warms up) and `problem` (Smoking / Obesity).

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath("."))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 185, "display.max_columns", 50)
import eda_analysis
from eda_analysis import stats, behavior, training, pref, figures, plots

# ╔═══ VIEW — the one knob ════════════════════════════════════════════════════════╗
# "all" = every arm | "L0" = K=0 arms (PTO_LA0/GRPO_LA0) | "L5" = K=5 arms (thin, paused).
# Sets BOTH the arm filter AND the results root -> results/<VIEW>/figures|tables/<group>/.
# Edit the default for interactive use; render_views.py overrides it via the EDA_VIEW env var.
VIEW = os.environ.get("EDA_VIEW", "L0")

cfg = eda_analysis.EdaConfig(
    view=VIEW,                             # arm filter + results/<VIEW>/ root
    export_group="2_heterogeneity",        # topic family = this notebook's number
    selection="all",
    focus_arms=None, focus_metric="Q1Q2",
)
S = eda_analysis.notebook_setup(cfg)
TRAITS = ["cooperation_level", "problem"]   # persona traits to split by (see PERSONA_COLS for more)

## 1 · Combined overview — all metrics per trait  `[EVAL]`
The single-glance companion to the per-metric subfolder below: one figure per trait with **rows = metric, cols = arm**, each cell split by persona category. Mirrors `1_Outcomes/trajectories_all_metrics`. → `2_heterogeneity/<trait>_all_metrics.png`.

In [ ]:
for trait in TRAITS:
    figo = plots.heterogeneity_overview_grid(S.SCORES, trait, arms=cfg.focus_arms, metrics=S.METRICS)
    if figo is None:
        print(f"{trait}: nothing plottable."); continue
    eda_analysis.save_fig(
        figo, f"{trait}_all_metrics",
        caption=f"All metrics by true patient {trait} (rows=metric, cols=arm), each cell split by persona "
                f"category — the combined overview companion to the per-metric {trait}/ subfolder.")
    plt.show()

## 2 · Every metric × every trait  `[EVAL]`
One small-multiples figure per (trait, metric): the metric across iterations, split by trait category (readable names, colourblind palette), a panel per arm. → `2_heterogeneity/<trait>/<metric>.png`.

In [ ]:
for trait in TRAITS:
    for m in S.METRICS:
        fig = plots.heterogeneity_grid(S.SCORES, trait, arms=cfg.focus_arms, metric=m)
        if fig is None:
            print(f"{trait} x {m}: nothing plottable."); continue
        eda_analysis.save_fig(
            fig, m, group=f"2_heterogeneity/{trait}",
            caption=f"{eda_analysis.display_label(m)} across iterations split by true patient {trait}; one panel per arm.")
        plt.show()

## 3 · Endpoint bars — where does the final gap concentrate?  `[EVAL]`
Final-iteration score per trait category × arm (dotted = base). **Read:** GRPO's late regression concentrates on the *Resistant* (Low-cooperation) personas while PTO holds above base.

In [ ]:
for trait in TRAITS:
    figb = plots.subgroup_endpoint_bars(S.SCORES, trait, arms=cfg.focus_arms,
                                        metric=cfg.focus_metric, palette=S.PALETTE)
    if figb is None:
        print(f"{trait}: nothing plottable."); continue
    eda_analysis.save_fig(
        figb, f"subgroup_endpoint_{trait}",
        caption=f"Final-iteration {eda_analysis.display_label(cfg.focus_metric)} by true patient {trait}, per arm (dotted = base) — where the endpoint gap concentrates.")
    plt.show()

## 4 · Artifact index
Refresh the per-view `results/<VIEW>/INDEX.md` (every notebook ends with this).

In [ ]:
print("index ->", eda_analysis.build_index())